# 🔬 D-FINE Single Line Diagram (SLD) Symbol Detection
### Interactive Training, Evaluation & DETR Box Refinement Notebook

This notebook configures and runs **D-FINE** (DETR architecture with Fine-Grained Distribution Refinement) directly on native COCO annotations.

### 1. Environment & Setup

In [ ]:
import os
import json
from pathlib import Path
import torch

base_dir = Path(".").resolve()
coco_dataset_dir = base_dir / "SLD ANNOTATION.v2-version-2.coco"
output_dir = base_dir / "results" / "dfine"
output_dir.mkdir(parents=True, exist_ok=True)

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device.upper()}")
print(f"COCO Dataset Path: {coco_dataset_dir}")

### 2. Verify COCO Annotation Schema

In [ ]:
with open(coco_dataset_dir / "train" / "_annotations.coco.json", 'r') as f:
    coco_train = json.load(f)

print("Categories in COCO dataset:")
for cat in coco_train['categories']:
    print(f"  ID {cat['id']}: {cat['name']}")

print(f"\nTotal Train Images: {len(coco_train['images'])}")
print(f"Total Train Annotations: {len(coco_train['annotations'])}")

### 3. Generate D-FINE Custom Config

In [ ]:
dfine_config = {
    "task": "detection",
    "num_classes": 2,
    "remap_mscoco_category": False,
    "train_dataloader": {
        "img_folder": str(coco_dataset_dir / "train"),
        "ann_file": str(coco_dataset_dir / "train" / "_annotations.coco.json"),
        "batch_size": 2
    },
    "val_dataloader": {
        "img_folder": str(coco_dataset_dir / "test"),
        "ann_file": str(coco_dataset_dir / "test" / "_annotations.coco.json"),
        "batch_size": 2
    }
}

config_file = output_dir / "dfine_custom_sld.json"
with open(config_file, 'w') as f:
    json.dump(dfine_config, f, indent=2)
print(f"Created D-FINE config at: {config_file}")

### 4. Clone D-FINE Engine & Train

In [ ]:
dfine_repo = base_dir / "D-FINE"
if not dfine_repo.exists():
    print("Cloning D-FINE repository...")
    !git clone https://github.com/Peterande/D-FINE.git

print("Ready to launch D-FINE Training!")